# Direction A atlas — run all events on Colab

Runs the full IC-optimization pipeline (`heatwave_ic.pipeline.run_event`) over every event config in `configs/`, in validation-first order (PNW → St. John's → new zones), and collects the cross-zone summary in `data/atlas_summary.csv`.

**Before running:** Runtime → Change runtime type → **GPU (A100 or L4, High-RAM)**.

**Session budgeting:** each event is ~75 Adam iterations over a 7–11-day unroll — budget roughly 1–2 events per Colab session. The runner is resume-aware: completed events are skipped, and with Drive persistence (cell 3) finished runs survive session death, so just re-run this notebook until the atlas is complete.

In [1]:
REPO_URL = "https://github.com/ieadoboe/heatwave-initial-conditions.git"

import shutil, sys
from pathlib import Path

name = Path(REPO_URL).stem
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / name]
root = next((p for p in candidates if (p / "heatwave_ic").is_dir()), None)
if root is None and "google.colab" in sys.modules:
    shutil.rmtree(name, ignore_errors=True)   # clear stale/partial clones
    !git clone {REPO_URL}
    root = Path.cwd() / name
if root is None or not (root / "heatwave_ic").is_dir():
    raise RuntimeError("heatwave_ic/ not found — clone failed or package not pushed.")
%cd {root}
sys.path.insert(0, str(root))
if "google.colab" in sys.modules:
    %pip install -q -U neuralgcm dinosaur gcsfs optax tqdm pyyaml zarr

Cloning into 'heatwave-initial-conditions'...
remote: Enumerating objects: 245, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 245 (delta 45), reused 52 (delta 28), pack-reused 171 (from 1)
Receiving objects: 100% (245/245), 169.24 MiB | 18.62 MiB/s, done.
Resolving deltas: 100% (127/127), done.
/content/heatwave-initial-conditions
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Persistence: completed runs sync to Drive after EVERY event, and are
# restored at the start of the next session. Set PERSIST = False to skip
# (outputs then die with the VM). IC zarrs are never persisted (tens of GB)
# — they rebuild automatically when a session needs one.
PERSIST = True
PERSIST_DIR = "/content/drive/MyDrive/heatwave_atlas"

if PERSIST and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
persist_flag = f"--persist-dir {PERSIST_DIR}" if PERSIST else ""

Mounted at /content/drive


## Run the atlas

The default runs everything not yet completed. To run a chosen subset this session (recommended: 1–2 events), use the second form.

In [3]:
!python scripts/run_atlas.py {persist_flag}

# Subset form — e.g. just the PNW validation run:
# !python scripts/run_atlas.py {persist_flag} --configs configs/pnw_jun2021.yaml

# Force a redo of an event (ignores its existing run dir):
# !python scripts/run_atlas.py {persist_flag} --rerun --configs configs/pnw_jun2021.yaml

Restoring completed runs from /content/drive/MyDrive/heatwave_atlas ...

pnw_jun2021: target (50.231, -121.581°E), event 2021-06-25 → 2021-07-01 (peak 2021-06-29)
  init 2021-06-20 (lead 9 d), evolve 11 d, target window = last 5 d
  beta=10 lambda=20 T_ref=298.15 lr=1e-09 iters=75
  model v1_precip/stochastic_precip_2_8_deg.pkl  IC data/era5_ic_pnw_jun2021_2021-06-20.zarr
Loading model v1_precip/stochastic_precip_2_8_deg.pkl ...
Opening ARCO-ERA5 and slicing the IC window at 2021-06-20 ...
Materialising 25 snapshots -> data/era5_ic_pnw_jun2021_2021-06-20.zarr ...
/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
IC built.

Plot -> plots/pnw_jun2021_opt_loss.pdf
Plot -> plots/pnw_jun2021_storyline.pdf
-> ok  gain=2.91 C

stjohns_aug2025: target (47.562, -52.713°E), event 2

In [4]:
import pandas as pd

summary = pd.read_csv("data/atlas_summary.csv")
display(summary)

print("\nFigures written to plots/:")
for p in sorted(Path("plots").glob("*_storyline.pdf")):
    print(" ", p)

,event,zone,koppen_point,koppen_cell,init_date,evol_days,run_dir,status,final_loss,final_box_T_C,final_reg,storyline_gain_C
0,pnw_jun2021,maritime coastal,Dsb,Dfb,2021-06-20,11.0,data/opt_runs/pnw_jun2021_lr1em09_it75_lam20_b...,ok,171.3584,30.44,0.012152,2.91
1,stjohns_aug2025,maritime coastal,Dfc,Dfb,2025-08-06,9.0,data/opt_runs/stjohns_aug2025_lr5em09_it75_lam...,ok,172.9823,21.79,0.022868,1.04
2,moscow_jul2010,continental interior,Dfb,Dfb,2010-07-23,9.0,data/opt_runs/moscow_jul2010_lr5em09_it75_lam1...,ok,173.7605,29.34,0.052085,2.34
3,japan_jul2018,subtropical humid,Cfa,Cfa,2018-07-17,7.0,data/opt_runs/japan_jul2018_lr5em09_it75_lam10...,ok,173.0905,29.11,0.004483,3.35
4,sahel_apr2024,arid,BSh,BSh,2024-03-28,8.0,data/opt_runs/sahel_apr2024_lr5em09_it75_lam10...,ok,174.7243,35.72,0.005244,2.78
5,brazil_nov2023,tropical,Aw,Aw,2023-11-08,11.0,data/opt_runs/brazil_nov2023_lr5em09_it75_lam1...,ok,173.7481,26.52,0.015163,1.52
6,siberia_jun2020,polar/high-latitude,Dfd,Dfd,2020-06-14,11.0,data/opt_runs/siberia_jun2020_lr5em09_it75_lam...,ok,165.0303,29.92,0.006612,6.70



Figures written to plots/:
  plots/brazil_nov2023_storyline.pdf
  plots/japan_jul2018_storyline.pdf
  plots/moscow_jul2010_storyline.pdf
  plots/pnw_jun2021_storyline.pdf
  plots/sahel_apr2024_storyline.pdf
  plots/siberia_jun2020_storyline.pdf
  plots/stjohns_aug2025_storyline.pdf


## Reading the result

- **`storyline_gain_C`** is the headline number per event: how much hotter the optimized-IC storyline peaks vs the unperturbed forecast (W&DL's PNW benchmark: **+3.7 °C**). If `pnw_jun2021` doesn't land near that, fix the pipeline before interpreting the other zones.
- Per-event outputs are in `data/opt_runs/<run-name>/`: `optimized.nc` / `original.nc` trajectories, initial-state fields (`*_opt.npy`, `*_original.npy`) for the sensitivity-map analysis, `losses.npy`, `storyline.csv`.
- Cross-zone comparison of the optimal-perturbation *structure* (the actual Direction A science) starts from those saved state fields.